# 🎙️ เสียงพากย์ — vox-2 "อำเภอเดียวกัน ห่างกันสี่พันปี"

**ใช้ยังไง:** Runtime → Change runtime type → **T4 GPU** → แล้ว Runtime → **Run all**

ได้ผลลัพธ์ 2 ไฟล์ดาวน์โหลดลงเครื่อง:
- **`voice.wav`** — เสียงทั้งคลิป 44 วินาที วางท่อนไว้ตรงตำแหน่งแล้ว → **ลากลง CapCut ที่ 0:00 ตรง ๆ จบ**
- `voice_segments.zip` — ไฟล์แยก 6 ท่อน เผื่ออยากขยับเอง

ใช้เสียงโคลนประจำช่องจาก `voice_profile.json` เหมือนที่รันบน RunPod ทุกอย่าง

In [ ]:
# 1) ติดตั้ง (~2 นาที)
!pip install -q omnivoice edge-tts soundfile pythainlp
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '❌ ไม่มี GPU — ไปที่ Runtime > Change runtime type > T4 GPU แล้วรันใหม่')

In [ ]:
# 2) ดึง repo (ได้โปรไฟล์เสียง + ไฟล์เสียงอ้างอิง + ตัวแปลงตัวเลขไทย)
import glob, json, os, sys
BRANCH = 'claude/tts-ai-tools-resources-m5rku5'
!rm -rf repo.zip claude-project1-*
!wget -q -O repo.zip https://codeload.github.com/suthipongs10-a11y/claude-project1/zip/refs/heads/{BRANCH}
!unzip -q repo.zip

TTS_DIR = glob.glob('claude-project1-*/tts-ai')[0]
sys.path.insert(0, TTS_DIR)
from normalize_th import normalize_th
PROFILE = json.load(open(f'{TTS_DIR}/voice_profile.json', encoding='utf-8'))
print('โมเดล:', PROFILE['model_id'], '| เวอร์ชันเสียง:', PROFILE['voice_version'])

In [ ]:
# 3) เสียงอ้างอิง — ต้องเป็นเสียงจริงเท่านั้น ถ้าไม่มีให้หยุด (กันไม่ให้ได้เสียงสำรองมาโดยไม่รู้ตัว)
REF_AUDIO = f"{TTS_DIR}/{PROFILE['ref_audio']}"
REF_TEXT_F = f"{TTS_DIR}/{PROFILE['ref_text_file']}"
assert os.path.exists(REF_AUDIO), f'ไม่พบ {REF_AUDIO}'
assert os.path.exists(REF_TEXT_F), f'ไม่พบ {REF_TEXT_F} — เสียงจริงยังไม่ครบคู่'
REF_TEXT = open(REF_TEXT_F, encoding='utf-8').read().strip()
print('✅ เสียงอ้างอิง:', PROFILE['voice_version'])

In [ ]:
# 4) บทพากย์ + ความยาวเป้าหมาย
#    'seconds' คือความยาวที่คลิปถูกตัดรอไว้แล้ว — ล็อกไว้เพื่อให้ voice.wav ตรงกับวิดีโอพอดี
#    ตัวเลขเขียนดิบได้ normalize_th() แปลงเป็นคำอ่านไทยให้เอง
SEGMENTS = [{"id": "s01_hook", "text": "สองสิ่งนี้ เกิดขึ้นในอำเภอเดียวกัน ห่างกันสี่พันปี", "start": 0.5, "seconds": 3.44}, {"id": "s02_origin", "text": "เมื่อ 2000 ปีก่อนคริสตกาล คนที่นี่ปลูกข้าวเป็นแล้ว เก่าที่สุดเท่าที่มีหลักฐานยืนยันในไทย", "start": 4.54, "seconds": 6.16}, {"id": "s03_accident", "text": "ค.ศ. 1945 ในอำเภอเดียวกันนั้น ชาวนาคนหนึ่งได้กลิ่นหอมมะลิ จากต้นข้าวในนาของเขา เขาเก็บเมล็ดนั้นไว้", "start": 11.3, "seconds": 8.68}, {"id": "s04_selection", "text": "จากรวงข้าว 199 รวง แถวที่ 105 คือผู้ชนะ ค.ศ. 1959 มันได้ชื่อว่า ข้าวขาวดอกมะลิ 105", "start": 20.58, "seconds": 9.36}, {"id": "s05_crown", "text": "ข้าวเมล็ดนั้น พาไทยขึ้นเป็นผู้ส่งออกข้าวอันดับหนึ่งของโลก และครองไว้ได้ 31 ปี", "start": 30.54, "seconds": 6.16}, {"id": "s06_twist", "text": "วันนี้ ไทยอยู่อันดับสาม อินเดียส่งออกมากกว่าเราสามเท่า", "start": 37.3, "seconds": 4.12}]
TOTAL_SEC = 44.0
for s in SEGMENTS:
    print(f"{s['id']:<14} เริ่ม {s['start']:>6.2f}s  ยาว {s['seconds']:>5.2f}s")

In [ ]:
# 5) โหลดโมเดล (~3-4 นาทีครั้งแรก)
import soundfile as sf, numpy as np
from omnivoice import OmniVoice
dev = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dt  = torch.float16 if torch.cuda.is_available() else torch.float32
model = OmniVoice.from_pretrained(PROFILE['model_id'], device_map=dev, dtype=dt)
SR = PROFILE.get('sample_rate', 24000)
print('พร้อม |', dev)

In [ ]:
# 6) เจนเสียงทีละท่อน
#    ส่ง duration= เพื่อบังคับความยาวให้ตรงกับที่คลิปตัดไว้ ถ้าเวอร์ชันโมเดลไม่รองรับ
#    จะถอยไปเจนแบบปกติแล้วแจ้งความยาวจริง (ถ้าไม่ตรง ส่งตารางท้ายเซลล์นี้ให้ Claude ขยับคิวให้)
os.makedirs('voice_out', exist_ok=True)
locked = True
for s in SEGMENTS:
    gen = normalize_th(s['text'])
    kw = dict(text=gen, ref_audio=REF_AUDIO, ref_text=REF_TEXT)
    try:
        audio = model.generate(duration=s['seconds'], **kw)
    except TypeError:
        locked = False
        audio = model.generate(**kw)
    a = np.asarray(audio[0], dtype='float32')
    sf.write(f"voice_out/{s['id']}.wav", a, SR)
    s['actual'] = round(len(a) / SR, 2)
    print(f"  {s['id']:<14} {s['actual']:>5.2f}s  (เป้า {s['seconds']:.2f}s)   {gen[:44]}")
print('\nล็อกความยาวได้' if locked else '\n⚠️ โมเดลไม่รับ duration — ความยาวจริงตามตารางข้างบน')

In [ ]:
# 7) รวมเป็นแทร็กเดียว 44 วินาที วางแต่ละท่อนตามเวลาเริ่มจริง
bed = np.zeros(int(TOTAL_SEC * SR), dtype='float32')
for s in SEGMENTS:
    a, _ = sf.read(f"voice_out/{s['id']}.wav", dtype='float32')
    i = int(round(s['start'] * SR))
    n = min(len(a), len(bed) - i)
    bed[i:i+n] += a[:n]
peak = float(np.max(np.abs(bed)))
if peak > 1.0: bed /= peak
sf.write('voice.wav', bed, SR)
print(f'voice.wav — {len(bed)/SR:.2f} วินาที  peak {peak:.2f}')

In [ ]:
# 8) 🎧 ฟังตรวจก่อนโหลด
from IPython.display import Audio, display, Markdown
display(Markdown('### เสียงอ้างอิง (ต้นแบบ)')); display(Audio(REF_AUDIO))
display(Markdown('### voice.wav — ทั้งคลิป 44 วินาที')); display(Audio('voice.wav'))

In [ ]:
# 9) ดาวน์โหลด
!zip -q -j voice_segments.zip voice_out/*.wav
from google.colab import files
files.download('voice.wav')
files.download('voice_segments.zip')

---
## เอาไปใช้ใน CapCut

1. ลากวิดีโอ `Vox2ThaiRice.mp4` (44 วินาที) ลงไทม์ไลน์
2. ลาก **`voice.wav`** ลงแทร็กเสียง **ให้เริ่มที่ 0:00**
3. เสร็จ — ทุกท่อนอยู่ตรงจังหวะภาพแล้ว ไม่ต้องขยับอะไร

ถ้าเซลล์ 6 ขึ้นคำเตือนว่าล็อกความยาวไม่ได้ ให้ copy ตารางในเซลล์นั้นส่งให้ Claude — จะขยับคิวในคลิปแล้วเรนเดอร์ใหม่ให้ตรงเสียง